# 02 — Preprocesamiento y Split de Datos
## Proyecto: Predicción de Abandono Estudiantil — EAFIT 2026-1
**Integrante:** Valeria Sague (EDA & Preprocesamiento)  
**Objetivo:** Limpiar el dataset, crear variable objetivo binaria, escalar features y generar el split train/val/test sin data leakage.  
Los archivos procesados serán usados por el Integrante 2 para modelado.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas')

✓ Librerías cargadas


In [2]:
# CORRECCIÓN: separador es coma, no punto y coma
df = pd.read_csv('../data/raw/dataset.csv', sep=',')

print(f'Shape original: {df.shape}')
print(f'\nDistribución target original:')
print(df['Target'].value_counts())
df.head()

Shape original: (4424, 35)

Distribución target original:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [3]:
# Binarizamos: Dropout=1, No Dropout (Graduate + Enrolled)=0
# Justificación: queremos detectar quién va a desertar (problema binario)
df['Target_Binary'] = (df['Target'] == 'Dropout').astype(int)

print('Variable objetivo binaria creada:')
print(f'  1 (Dropout):    {df["Target_Binary"].sum()} ({df["Target_Binary"].mean()*100:.1f}%)')
print(f'  0 (No Dropout): {(1-df["Target_Binary"]).sum()} ({(1-df["Target_Binary"]).mean()*100:.1f}%)')
print(f'\nJustificación: problema de clasificación binaria.')
print('Nos interesa identificar a quienes van a desertar para intervenir a tiempo.')

Variable objetivo binaria creada:
  1 (Dropout):    1421 (32.1%)
  0 (No Dropout): 3003 (67.9%)

Justificación: problema de clasificación binaria.
Nos interesa identificar a quienes van a desertar para intervenir a tiempo.


In [4]:
print('Verificación de nulos post-transformación:')
print(f'Nulos totales: {df.isnull().sum().sum()}')
print('✓ Sin valores nulos. No se requiere imputación.')

Verificación de nulos post-transformación:
Nulos totales: 0
✓ Sin valores nulos. No se requiere imputación.


In [5]:
print('=' * 50)
print('ESTRATEGIA DE OUTLIERS')
print('=' * 50)
print('\nDecisión: MANTENER todos los outliers.')
print('Justificación:')
print('  1. Son valores académicamente válidos (0 materias aprobadas = reprobó todo)')
print('  2. LightGBM es robusto a outliers (basado en árboles)')
print('  3. Eliminarlos distorsionaría el perfil real de estudiantes en riesgo')
print('  4. Los casos extremos son precisamente los más importantes para detectar')

ESTRATEGIA DE OUTLIERS

Decisión: MANTENER todos los outliers.
Justificación:
  1. Son valores académicamente válidos (0 materias aprobadas = reprobó todo)
  2. LightGBM es robusto a outliers (basado en árboles)
  3. Eliminarlos distorsionaría el perfil real de estudiantes en riesgo
  4. Los casos extremos son precisamente los más importantes para detectar


In [6]:
# X: todas las features excepto las columnas de target
X = df.drop(columns=['Target', 'Target_Binary'])
y = df['Target_Binary']

print(f'Features (X): {X.shape[1]} columnas')
print(f'Samples:      {X.shape[0]} filas')
print(f'\nColumnas en X:')
print(list(X.columns))

Features (X): 34 columnas
Samples:      4424 filas

Columnas en X:
['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation rate', 'GDP']


In [7]:
# Split 70% train / 15% val / 15% test
# stratify=y garantiza que la proporción de desertores se mantiene en cada split
# random_state=42 para reproducibilidad

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('Split estratificado completado:')
print(f'  Train: {X_train.shape[0]} filas | Dropout: {y_train.mean()*100:.1f}%')
print(f'  Val:   {X_val.shape[0]} filas   | Dropout: {y_val.mean()*100:.1f}%')
print(f'  Test:  {X_test.shape[0]} filas  | Dropout: {y_test.mean()*100:.1f}%')
print(f'\n✓ Las proporciones de dropout se mantienen en los 3 splits (sin data leakage)')

Split estratificado completado:
  Train: 3096 filas | Dropout: 32.1%
  Val:   664 filas   | Dropout: 32.2%
  Test:  664 filas  | Dropout: 32.1%

✓ Las proporciones de dropout se mantienen en los 3 splits (sin data leakage)


In [8]:
# CRÍTICO: el scaler se ajusta SOLO en train
# Luego se aplica (transform) en val y test sin re-ajustar
# Si ajustáramos en todo el dataset habría data leakage

scaler = StandardScaler()

# fit_transform SOLO en train
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Solo transform en val y test
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print('Escalado con StandardScaler:')
print('  ✓ fit_transform aplicado SOLO en train')
print('  ✓ transform (sin fit) aplicado en val y test')
print('  ✓ Sin data leakage')
print(f'\nMedia post-escalado en train (debe ser ~0): {X_train_scaled.mean().mean():.4f}')
print(f'Std post-escalado en train (debe ser ~1):  {X_train_scaled.std().mean():.4f}')

Escalado con StandardScaler:
  ✓ fit_transform aplicado SOLO en train
  ✓ transform (sin fit) aplicado en val y test
  ✓ Sin data leakage

Media post-escalado en train (debe ser ~0): -0.0000
Std post-escalado en train (debe ser ~1):  1.0002


In [9]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Guardar splits
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_val_scaled.to_csv('../data/processed/X_val.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Guardar scaler para uso posterior (Integrante 3 lo necesita para el agente)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Guardar nombres de features para referencia
pd.Series(X.columns.tolist()).to_csv('../data/processed/feature_names.csv', index=False)

print('✓ Archivos guardados en data/processed/:')
for archivo in sorted(os.listdir('../data/processed/')):
    if archivo.endswith('.csv'):
        ruta = f'../data/processed/{archivo}'
        size = os.path.getsize(ruta)
        print(f'  {archivo} ({size/1024:.1f} KB)')
print(f'\n✓ Scaler guardado en models/scaler.pkl')

✓ Archivos guardados en data/processed/:
  X_test.csv (438.3 KB)
  X_train.csv (2041.3 KB)
  X_val.csv (438.3 KB)
  feature_names.csv (0.8 KB)
  y_test.csv (2.0 KB)
  y_train.csv (9.1 KB)
  y_val.csv (2.0 KB)

✓ Scaler guardado en models/scaler.pkl


In [10]:
print('=' * 60)
print('RESUMEN COMPLETO DE PREPROCESAMIENTO')
print('=' * 60)
print(f'\nTarget:         Binario (1=Desertor, 0=No desertor)')
print(f'Nulos:          Ninguno — no se requirió imputación')
print(f'Outliers:       Mantenidos — valores académicamente válidos')
print(f'Escalado:       StandardScaler (fit solo en train)')
print(f'Split:          70/15/15 estratificado por clase')
print(f'random_state:   42 (reproducible)')
print(f'\nTamaños:')
print(f'  Train: {X_train_scaled.shape}')
print(f'  Val:   {X_val_scaled.shape}')
print(f'  Test:  {X_test_scaled.shape}')
print(f'\nDesbalance en train: {dict(y_train.value_counts())}')
print(f'El Integrante 2 debe usar class_weight=balanced en LightGBM')
print(f'\n✓ Preprocesamiento completo. El Integrante 2 ya puede empezar el modelado.')

RESUMEN COMPLETO DE PREPROCESAMIENTO

Target:         Binario (1=Desertor, 0=No desertor)
Nulos:          Ninguno — no se requirió imputación
Outliers:       Mantenidos — valores académicamente válidos
Escalado:       StandardScaler (fit solo en train)
Split:          70/15/15 estratificado por clase
random_state:   42 (reproducible)

Tamaños:
  Train: (3096, 34)
  Val:   (664, 34)
  Test:  (664, 34)

Desbalance en train: {0: np.int64(2102), 1: np.int64(994)}
El Integrante 2 debe usar class_weight=balanced en LightGBM

✓ Preprocesamiento completo. El Integrante 2 ya puede empezar el modelado.
